In [ ]:
import pandas as pd
import numpy as np


In [ ]:
import mcf


In [ ]:
from mcf.mcf_main import ModifiedCausalForest


In [ ]:
df_aggregate = pd.read_excel('/Users/ghadaelhusseini/Documents/Kassel University Masters /Thesis/Replication/Lechner(2018)_Data/Data120190214.xlsx')

In [ ]:
columns_to_keep = [
    "case_id", "ADM2ID", "country_num", "year",  "CONFLICT_YES", "CONFLICT_YES_P1", "mean_light", "MEAN_LIGHT_LOG1",  "SUM_RESSOURCES", "PX", "PX_LOWVALW", "HIGHVALW" , 'PX_DISC_2', 'PX_DISC_3', 'PX_DISC_4', 'PX_DISC_5', "mine_count",
    "area", "tri", "min_elev", "max_elev", "median_elev", "distance_to_capital",
    "frac_LU_artificial", "frac_LU_cropland", "frac_LU_grass", "frac_LU_tree",
    "frac_LU_shrubs", "frac_LU_herbaceous", "frac_LU_mangroves", "frac_LU_sparse",
    "frac_LU_baresoil", "frac_LU_snow", "frac_LU_water",
    "temperature_mean", "precipitation_mean", "AgriSuit_mean",
    "ETH_ethnologue_groups", "ETH_Murdock_groups", "pre_col_polity",
    "ports", "port_oil_terminal", "pop", "pop_density",
    "icrg_qog", "p_xconst", "wbgi_cce", "wbgi_gee", "wbgi_pve",
    "wbgi_rle", "wbgi_rqe", "wbgi_vae", "p_polity2",
    "iaep_unitary_federal", "wdi_gdpcapcon2010", "fh_pr", "fh_cl",
    "ethfrac", "relfrac", "ht_colonial"
]

In [ ]:
df = df_aggregate[[col for col in columns_to_keep if col in df_aggregate.columns]]


In [ ]:
print(df.head())

In [ ]:

# Specify the folder path (change this to your folder)
folder_path = '/Users/ghadaelhusseini/Documents/Kassel University Masters /Thesis/my_Data'

# Save to Excel in that folder
df.to_excel(folder_path + "Filtered_df.xlsx", index=False)

print("Excel file saved successfully in:", folder_path)



In [ ]:
df = pd.read_excel('/Users/ghadaelhusseini/Documents/Kassel University Masters /Thesis/my_DataFiltered_df.xlsx')


In [ ]:
print("Rows:", len(df))
print("Columns available:", df.columns.tolist()[:50])

In [ ]:
outcome= "CONFLICT_YES"

In [ ]:
# -----------------------------
# STEP 1: Start with everyone as control (non-mining districts)
# -----------------------------
df["treat"] = 0

# -----------------------------
# STEP 2: Assign treatment bins to mining districts
# -----------------------------
# For districts with at least one mine, assign treatment value from PX_DISC_3
df.loc[df["mine_count"] > 0, "treat"] = df.loc[df["mine_count"] > 0, "PX_DISC_3"]

# -----------------------------
# Convert to integer type
df["treat"] = df["treat"].astype(int)

# -----------------------------
# STEP 4: Check the result
# -----------------------------
print("Unique treatment categories:", sorted(df["treat"].unique()))
print("\nTreatment group counts:")
print(df["treat"].value_counts().sort_index())


In [ ]:
print(df["treat"].describe())
print(df["treat"].value_counts().sort_index())


In [ ]:
x_ord = [
    "area",
    "tri",
    "min_elev", "max_elev", "median_elev",
    "distance_to_capital",
    "frac_LU_artificial", "frac_LU_cropland", "frac_LU_grass", "frac_LU_tree",
    "frac_LU_shrubs", "frac_LU_herbaceous", "frac_LU_mangroves", "frac_LU_sparse",
    "frac_LU_baresoil", "frac_LU_snow", "frac_LU_water",
    "temperature_mean", "precipitation_mean",
    "AgriSuit_mean",
    "pop", "pop_density",
    "icrg_qog", "p_xconst", "wbgi_cce", "wbgi_gee", "wbgi_pve",
    "wbgi_rle", "wbgi_rqe", "wbgi_vae",
    "p_polity2"
]

In [ ]:
x_unord = [
    "ports",
    "port_oil_terminal"
]

In [ ]:
help(ModifiedCausalForest)


In [ ]:
my_mcf = ModifiedCausalForest(
    var_y_name=outcome,  # Outcome variable
    var_d_name="treat",    # Treatment variable
    var_x_name_ord=x_ord,  # Ordered covariates
    #var_x_name_unord=x_unord,  # Unordered covariate
    _int_show_plots=False,
    cs_detect_const_vars_stop=False,
    cs_max_del_train=  # common support threshold
)

In [ ]:
x_ord = [v for v in x_ord if v not in ["frac_LU_snow"]]
x_unord = [v for v in x_unord if v not in ["port_oil_terminal_prime"]]


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
training_df, prediction_df = train_test_split(df, test_size=0.4, random_state=42)


In [ ]:
my_mcf.train(training_df)


In [ ]:
results = my_mcf.predict(prediction_df)


In [ ]:
ate_array = results.get('ate')
print("Average Treatment Effect (ATE):\n", ate_array)

In [ ]:
for column in df.columns:
    print(f"Descriptive statistics for {column}:")
    print(df[column].describe())  # gives count, mean, std, min, 25%, 50%, 75%, max
    print("\n")

In [ ]:
from mcf import ModifiedCausalForest
help(ModifiedCausalForest)
